In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, learning_curve
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family': 'DejaVu Serif',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linestyle': '--',
})

P = {'train':'#2166AC','test':'#D6604D','pred':'#1A7A4A','accent':'#F4A460'}

#Data & Features
df = pd.read_csv('/content/Dengue_Climate_Bangladesh.csv')
df = df.sort_values(['YEAR','MONTH']).reset_index(drop=True)

df['TEMP_RANGE']   = df['MAX'] - df['MIN']
df['TEMP_MEAN']    = (df['MAX'] + df['MIN']) / 2
df['HEAT_INDEX']   = df['TEMP_MEAN'] * df['HUMIDITY'] / 100
df['RAIN_LOG']     = np.log1p(df['RAINFALL'])
df['DENGUE_LAG1']  = np.log1p(df['DENGUE'].shift(1).fillna(0))
df['DENGUE_LAG2']  = np.log1p(df['DENGUE'].shift(2).fillna(0))
df['DENGUE_ROLL3'] = np.log1p(df['DENGUE'].shift(1).rolling(3,min_periods=1).mean().fillna(0))
df['SIN_MONTH']    = np.sin(2*np.pi*df['MONTH']/12)
df['COS_MONTH']    = np.cos(2*np.pi*df['MONTH']/12)

FEATURES = ['MIN','MAX','HUMIDITY','RAINFALL','TEMP_RANGE','TEMP_MEAN',
            'HEAT_INDEX','RAIN_LOG','DENGUE_LAG1','DENGUE_LAG2','DENGUE_ROLL3',
            'SIN_MONTH','COS_MONTH']
FEAT_LABELS = ['Min Temp','Max Temp','Humidity','Rainfall','Temp Range','Mean Temp',
               'Heat Index','Log Rainfall','log(Lag-1)','log(Lag-2)','log(Roll-3)',
               'sin(Month)','cos(Month)']

X = df[FEATURES].values
y_raw = df['DENGUE'].values
y_log = np.log1p(y_raw)   # log-transform target

#80/20 Split
X_tr, X_te, y_tr, y_te, yr_tr, yr_te = train_test_split(
    X, y_log, y_raw, test_size=0.20, random_state=42, shuffle=True)

sx = StandardScaler(); sy = StandardScaler()
X_tr_s = sx.fit_transform(X_tr)
X_te_s = sx.transform(X_te)
y_tr_s = sy.fit_transform(y_tr.reshape(-1,1)).ravel()

print(f"Samples: {len(df)}  |  Train: {len(X_tr)} (80%)  |  Test: {len(X_te)} (20%)")

#Grid Search
param_grid = {'C':[0.5,1,5,10,50,100,200],
              'gamma':['scale','auto',0.001,0.01,0.05,0.1],
              'epsilon':[0.01,0.05,0.1,0.2]}
gs = GridSearchCV(SVR(kernel='rbf'), param_grid, cv=5, scoring='r2', n_jobs=-1)
gs.fit(X_tr_s, y_tr_s)
bp = gs.best_params_
print(f"Best params: {bp}")

#Final Model
model = SVR(kernel='rbf', **bp)
model.fit(X_tr_s, y_tr_s)

def predict_cases(model, X_s, sy):
    y_log_pred = sy.inverse_transform(model.predict(X_s).reshape(-1,1)).ravel()
    return np.clip(np.expm1(y_log_pred), 0, None)

y_tr_pred = predict_cases(model, X_tr_s, sy)
y_te_pred = predict_cases(model, X_te_s, sy)

def metrics(actual, pred, label):
    r2   = r2_score(actual, pred)
    rmse = np.sqrt(mean_squared_error(actual, pred))
    mae  = mean_absolute_error(actual, pred)
    mask = actual > 0
    mape = np.mean(np.abs((actual[mask]-pred[mask])/actual[mask]))*100
    r    = np.corrcoef(actual, pred)[0,1]
    # on log scale
    r2_log  = r2_score(np.log1p(actual), np.log1p(pred))
    rmse_log= np.sqrt(mean_squared_error(np.log1p(actual), np.log1p(pred)))
    return dict(Set=label, R2=round(r2,4), RMSE=round(rmse,2), MAE=round(mae,2),
                MAPE=round(mape,2), r=round(r,4), R2_log=round(r2_log,4), RMSE_log=round(rmse_log,4))

m_tr = metrics(yr_tr, y_tr_pred, 'Train 80%')
m_te = metrics(yr_te, y_te_pred, 'Test  20%')
print("\n── Metrics (original scale) ──")
for m in [m_tr, m_te]: print(m)

# 10-fold CV on log scale
X_all_s = StandardScaler().fit_transform(X)
y_all_s = StandardScaler().fit_transform(y_log.reshape(-1,1)).ravel()
cv_r2 = cross_val_score(SVR(kernel='rbf',**bp), X_all_s, y_all_s, cv=10, scoring='r2')
print(f"\n10-Fold CV R² (log scale): {cv_r2.mean():.4f} ± {cv_r2.std():.4f}")

# FIG 1 : Scatter Actual vs Predicted (log scale + raw scale)
fig, axes = plt.subplots(1,2,figsize=(14,5.5))
fig.suptitle('SVR with RBF Kernel — Actual vs. Predicted\nDengue Cases, Bangladesh (2008–2025)',
             fontsize=14, fontweight='bold', y=1.02)

datasets = [
    (axes[0], yr_tr, y_tr_pred, f'Training Set (80%, n={len(yr_tr)})', P['train']),
    (axes[1], yr_te, y_te_pred, f'Test Set (20%, n={len(yr_te)})',     P['test']),
]
for ax, actual, pred, ttl, col in datasets:
    ax.scatter(actual, pred, alpha=0.65, s=50, color=col, edgecolors='white', lw=0.4, zorder=3)
    lim_max = max(actual.max(), pred.max()) * 1.05
    ax.plot([0, lim_max],[0, lim_max], 'k--', lw=1.5, label='y = x (Perfect)')
    m_s, b_s, *_ = stats.linregress(actual, pred)
    xl = np.linspace(0, lim_max, 300)
    ax.plot(xl, m_s*xl+b_s, color='crimson', lw=1.8, label='Regression fit')
    r2 = r2_score(actual, pred)
    rmse = np.sqrt(mean_squared_error(actual, pred))
    mae  = mean_absolute_error(actual, pred)
    ax.set_xscale('symlog', linthresh=10); ax.set_yscale('symlog', linthresh=10)
    ax.set_xlabel('Actual Dengue Cases (log scale)')
    ax.set_ylabel('Predicted Dengue Cases (log scale)')
    ax.set_title(ttl, fontweight='bold')
    ax.legend(loc='upper left', framealpha=0.85)
    ax.text(0.97, 0.07, f'R² = {r2:.4f}\nRMSE = {rmse:,.0f}\nMAE = {mae:,.0f}',
            transform=ax.transAxes, ha='right', va='bottom', fontsize=10,
            bbox=dict(boxstyle='round,pad=0.4', fc='white', ec='grey', alpha=0.85))
plt.tight_layout()
plt.savefig('/content/fig1_actual_vs_predicted.png')
plt.close(); print("Fig 1 saved")

# FIG 2 : Time-series (chronological split for visualization)
cut = int(0.80 * len(df))
df_tr_c = df.iloc[:cut]; df_te_c = df.iloc[cut:]
X_tr_c_s = sx.transform(df_tr_c[FEATURES].values)
X_te_c_s = sx.transform(df_te_c[FEATURES].values)
y_tr_c_pred = predict_cases(model, X_tr_c_s, sy)
y_te_c_pred = predict_cases(model, X_te_c_s, sy)

ti_tr = np.arange(len(df_tr_c))
ti_te = np.arange(len(df_tr_c), len(df))

fig, ax = plt.subplots(figsize=(15,5))
ax.plot(ti_tr, df_tr_c['DENGUE'].values, color=P['train'], lw=1.4, label='Actual (Train 80%)')
ax.plot(ti_tr, y_tr_c_pred,              color=P['train'], lw=1.1, ls='--', alpha=0.55, label='Predicted (Train)')
ax.plot(ti_te, df_te_c['DENGUE'].values, color=P['test'],  lw=1.8, label='Actual (Test 20%)')
ax.plot(ti_te, y_te_c_pred,              color=P['pred'],  lw=2.0, ls='--', label='Predicted (Test)')
ax.axvline(cut-0.5, color='black', lw=1.2, ls=':', alpha=0.7)
ax.text(cut+0.5, ax.get_ylim()[1]*0.92, '← Train | Test →', fontsize=9)
ticks = list(range(0, len(df), 12))
ax.set_xticks(ticks)
ax.set_xticklabels([str(int(df.iloc[i]['YEAR'])) for i in ticks], rotation=45)
ax.set_xlabel('Year'); ax.set_ylabel('Monthly Dengue Cases')
ax.set_title('SVR (RBF Kernel) — Chronological Time-Series: Actual vs. Predicted\nBangladesh 2008–2025',
             fontweight='bold')
ax.legend(loc='upper left', framealpha=0.85)
r2_te_c = r2_score(df_te_c['DENGUE'].values, y_te_c_pred)
ax.text(0.99, 0.97, f'Test R² = {r2_te_c:.4f}', transform=ax.transAxes,
        ha='right', va='top', fontsize=10,
        bbox=dict(boxstyle='round',fc='white',ec='grey',alpha=0.85))
plt.tight_layout()
plt.savefig('/content/fig2_timeseries.png')
plt.close(); print("Fig 2 saved")

# FIG 3 : Residual Diagnostics (4-panel)
res = yr_te - y_te_pred   # raw residuals
res_std = (res - res.mean()) / res.std()

fig, axes = plt.subplots(2,2,figsize=(13,9))
fig.suptitle('SVR (RBF Kernel) — Residual Diagnostic Plots (Test Set)',
             fontsize=14, fontweight='bold')

ax = axes[0,0]
ax.scatter(y_te_pred, res, alpha=0.65, s=50, color=P['test'], edgecolors='white', lw=0.4)
ax.axhline(0, color='red', lw=1.8, ls='--')
ax.axhline(2*res.std(),  color='grey', lw=1, ls=':')
ax.axhline(-2*res.std(), color='grey', lw=1, ls=':')
ax.set_xlabel('Fitted Values'); ax.set_ylabel('Residuals')
ax.set_title('(a) Residuals vs. Fitted Values', fontweight='bold')

ax = axes[0,1]
(osm, osr), (slope, intercept, r_qq) = stats.probplot(res, dist='norm')
ax.scatter(osm, osr, alpha=0.65, s=45, color=P['test'], edgecolors='white', lw=0.4)
xl = np.linspace(min(osm), max(osm), 200)
ax.plot(xl, slope*xl+intercept, 'r--', lw=1.8)
ax.set_xlabel('Theoretical Quantiles'); ax.set_ylabel('Sample Quantiles')
ax.set_title('(b) Normal Q–Q Plot of Residuals', fontweight='bold')
ax.text(0.05,0.92,f'Pearson r = {r_qq:.3f}', transform=ax.transAxes, fontsize=9,
        bbox=dict(boxstyle='round',fc='white',ec='grey',alpha=0.85))

ax = axes[1,0]
ax.hist(res, bins=22, color=P['train'], edgecolor='white', alpha=0.85, density=True)
xk = np.linspace(res.min(), res.max(), 300)
ax.plot(xk, stats.norm.pdf(xk, res.mean(), res.std()), 'r-', lw=2, label='Normal PDF')
ax.set_xlabel('Residuals'); ax.set_ylabel('Density')
ax.set_title('(c) Distribution of Residuals', fontweight='bold')
ax.legend()
sw_stat, sw_p = stats.shapiro(res[:50]) if len(res)>50 else stats.shapiro(res)
ax.text(0.97,0.92,f'Shapiro–Wilk\np = {sw_p:.3f}',
        transform=ax.transAxes, ha='right', fontsize=9,
        bbox=dict(boxstyle='round',fc='white',ec='grey',alpha=0.85))

ax = axes[1,1]
ax.plot(range(len(res_std)), res_std, 'o-', color=P['test'], ms=4.5, lw=0.9, alpha=0.75)
ax.axhline(0,  color='red',  lw=1.8, ls='--')
ax.axhline(2,  color='grey', lw=1,   ls=':')
ax.axhline(-2, color='grey', lw=1,   ls=':')
ax.fill_between(range(len(res_std)), -2, 2, alpha=0.07, color='grey')
ax.set_xlabel('Observation Index'); ax.set_ylabel('Standardised Residuals')
ax.set_title('(d) Standardised Residuals vs. Index', fontweight='bold')

plt.tight_layout()
plt.savefig('/content/fig3_residuals.png')
plt.close(); print("Fig 3 saved")

# FIG 4 : Permutation Feature Importance
perm = permutation_importance(model, X_te_s, y_te, n_repeats=30, random_state=42, scoring='r2')
imp_m = perm.importances_mean
imp_s = perm.importances_std
order = np.argsort(imp_m)

fig, ax = plt.subplots(figsize=(9,7))
cols = [P['train'] if v>=0 else '#C0392B' for v in imp_m[order]]
ax.barh(np.array(FEAT_LABELS)[order], imp_m[order], xerr=imp_s[order],
        color=cols, edgecolor='white', height=0.65,
        error_kw=dict(ecolor='#444', capsize=3, lw=1))
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Mean Decrease in R² (Permutation Importance, Test Set)')
ax.set_title('SVR (RBF Kernel) — Permutation Feature Importance\n(30 Repeats, Test Set)',
             fontweight='bold')
for i,(v,e) in enumerate(zip(imp_m[order], imp_s[order])):
    ax.text(max(0,v)+e+0.001, i, f'{v:.4f}', va='center', fontsize=8.5)
plt.tight_layout()
plt.savefig('/content/fig4_feature_importance.png')
plt.close(); print("Fig 4 saved")

# FIG 5 : Learning Curve
tr_sz, tr_sc, va_sc = learning_curve(
    SVR(kernel='rbf',**bp), X_all_s, y_all_s,
    train_sizes=np.linspace(0.1,1.0,10), cv=5, scoring='r2', n_jobs=-1)

fig, ax = plt.subplots(figsize=(9,5))
ax.plot(tr_sz, tr_sc.mean(1), 'o-', color=P['train'], lw=2, label='Training Score')
ax.fill_between(tr_sz, tr_sc.mean(1)-tr_sc.std(1), tr_sc.mean(1)+tr_sc.std(1), alpha=0.15, color=P['train'])
ax.plot(tr_sz, va_sc.mean(1), 's-', color=P['test'],  lw=2, label='Cross-Validation Score')
ax.fill_between(tr_sz, va_sc.mean(1)-va_sc.std(1), va_sc.mean(1)+va_sc.std(1), alpha=0.15, color=P['test'])
ax.set_xlabel('Training Sample Size'); ax.set_ylabel('R² Score (log scale)')
ax.set_title('SVR (RBF) — Learning Curve (5-Fold CV, Log-Transformed Target)',
             fontweight='bold')
ax.legend(framealpha=0.85)
plt.tight_layout()
plt.savefig('/content/fig5_learning_curve.png')
plt.close(); print("Fig 5 saved")

# FIG 6 : Error Distribution — Train vs Test
fig, axes = plt.subplots(1,2,figsize=(13,5))
fig.suptitle('SVR (RBF) — Prediction Error Distribution (Raw Scale)', fontsize=14, fontweight='bold')
for ax, actual, pred, label, col in [
        (axes[0], yr_tr, y_tr_pred, f'Training Set (80%, n={len(yr_tr)})', P['train']),
        (axes[1], yr_te, y_te_pred, f'Test Set (20%, n={len(yr_te)})',     P['test'])]:
    err = actual - pred
    ax.hist(err, bins=22, color=col, edgecolor='white', alpha=0.8, density=True)
    xk = np.linspace(err.min(), err.max(), 300)
    ax.plot(xk, stats.norm.pdf(xk, err.mean(), err.std()), 'r-', lw=2, label='Normal PDF')
    ax.set_xlabel('Prediction Error (Actual − Predicted)')
    ax.set_ylabel('Density')
    ax.set_title(label, fontweight='bold')
    ax.legend()
    ax.text(0.97,0.92,f'Mean = {err.mean():.1f}\nSD = {err.std():.1f}',
            transform=ax.transAxes, ha='right', fontsize=9,
            bbox=dict(boxstyle='round',fc='white',ec='grey',alpha=0.85))
plt.tight_layout()
plt.savefig('/content/fig6_error_distribution.png')
plt.close(); print("Fig 6 saved")

# FIG 7 : Test Set Bar Chart (Actual vs Predicted per sample)
_, te_idx = train_test_split(np.arange(len(df)), test_size=0.20, random_state=42, shuffle=True)
te_idx_sorted = sorted(te_idx)
te_actual  = df.iloc[te_idx_sorted]['DENGUE'].values
te_pred_arr = predict_cases(model, sx.transform(df.iloc[te_idx_sorted][FEATURES].values), sy)
pct_e = np.where(te_actual==0, np.nan, np.abs((te_actual-te_pred_arr)/te_actual)*100)

result_df = pd.DataFrame({
    'Year':      df.iloc[te_idx_sorted]['YEAR'].values.astype(int),
    'Month':     df.iloc[te_idx_sorted]['MONTH'].values.astype(int),
    'Actual':    te_actual,
    'Predicted': np.round(te_pred_arr,1),
    'Abs_Error': np.round(np.abs(te_actual-te_pred_arr),1),
    'Pct_Error': np.round(pct_e,1),
}).sort_values(['Year','Month'])
result_df.to_csv('/content/test_predictions_table.csv', index=False)

x = np.arange(len(result_df)); w = 0.42
fig, ax = plt.subplots(figsize=(16,5.5))
ax.bar(x-w/2, result_df['Actual'],    width=w, color=P['test'],  alpha=0.88, label='Actual')
ax.bar(x+w/2, result_df['Predicted'], width=w, color=P['pred'],  alpha=0.88, label='SVR Predicted')
ax.set_xticks(x[::3])
ax.set_xticklabels([f"{int(r.Year)}-{int(r.Month):02d}" for _,r in result_df.iloc[::3].iterrows()],
                   rotation=45, ha='right', fontsize=8)
ax.set_xlabel('Year–Month'); ax.set_ylabel('Dengue Cases')
ax.set_title(f'SVR (RBF) — Test Set Actual vs. Predicted (n={len(result_df)}, 20% of data)',
             fontweight='bold')
ax.legend(framealpha=0.85)
r2_b = r2_score(result_df['Actual'], result_df['Predicted'])
rmse_b = np.sqrt(mean_squared_error(result_df['Actual'], result_df['Predicted']))
ax.text(0.99,0.97,f'R² = {r2_b:.4f} | RMSE = {rmse_b:,.0f}',
        transform=ax.transAxes, ha='right', va='top', fontsize=10,
        bbox=dict(boxstyle='round',fc='white',ec='grey',alpha=0.85))
plt.tight_layout()
plt.savefig('/content/fig7_test_bar_chart.png')
plt.close(); print("Fig 7 saved")

# FIG 8 : Correlation Heatmap
hmap_cols = ['MIN','MAX','HUMIDITY','RAINFALL','TEMP_RANGE','TEMP_MEAN',
             'HEAT_INDEX','RAIN_LOG','DENGUE_LAG1','DENGUE_LAG2','DENGUE_ROLL3','DENGUE']
hmap_lbls = ['Min T','Max T','Humidity','Rainfall','T Range','Mean T',
             'Heat Idx','Log Rain','log(Lag-1)','log(Lag-2)','log(Roll-3)','Dengue']
cm = df[hmap_cols].corr()

fig, ax = plt.subplots(figsize=(11,9))
mask = np.triu(np.ones_like(cm, dtype=bool))
sns.heatmap(cm, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.4, linecolor='white',
            ax=ax, cbar_kws={'shrink':0.78},
            xticklabels=hmap_lbls, yticklabels=hmap_lbls)
ax.set_title('Pearson Correlation Matrix — Features and Target Variable\n(Bangladesh Dengue Dataset, 2008–2025)',
             fontweight='bold')
plt.tight_layout()
plt.savefig('/content/fig8_correlation_heatmap.png')
plt.close(); print("Fig 8 saved")

# FIG 9 : 10-Fold CV (Boxplot + Bar)
fig, axes = plt.subplots(1,2,figsize=(12,5))
fig.suptitle('SVR (RBF) — 10-Fold Cross-Validation (Log-Transformed Target)',
             fontsize=14, fontweight='bold')

ax = axes[0]
bp2 = ax.boxplot(cv_r2, vert=True, patch_artist=True,
                 medianprops=dict(color='red',lw=2),
                 whiskerprops=dict(lw=1.5),
                 capprops=dict(lw=1.5))
bp2['boxes'][0].set_facecolor(P['train']); bp2['boxes'][0].set_alpha(0.75)
ax.scatter([1]*len(cv_r2), cv_r2, color=P['test'], s=55, zorder=5, edgecolors='white', lw=0.5)
ax.set_xticks([1]); ax.set_xticklabels(['SVR–RBF'])
ax.set_ylabel('R² Score'); ax.set_title('(a) R² Distribution across Folds', fontweight='bold')
ax.text(0.97,0.06,f'Mean = {cv_r2.mean():.4f}\nSD = {cv_r2.std():.4f}',
        transform=ax.transAxes, ha='right', fontsize=10,
        bbox=dict(boxstyle='round',fc='white',ec='grey',alpha=0.85))

ax = axes[1]
ax.bar(range(1,11), cv_r2, color=P['train'], alpha=0.82, edgecolor='white')
ax.axhline(cv_r2.mean(), color='red', lw=1.8, ls='--', label=f'Mean R² = {cv_r2.mean():.4f}')
ax.set_xlabel('Fold'); ax.set_ylabel('R² Score')
ax.set_title('(b) R² per Fold', fontweight='bold')
ax.set_xticks(range(1,11)); ax.legend(framealpha=0.85)
plt.tight_layout()
plt.savefig('/content/fig9_cross_validation.png')
plt.close(); print("Fig 9 saved")

# FIG 10 : Seasonal Pattern (Monthly Mean)
all_pred = predict_cases(model, sx.transform(X), sy)
df['SVR_Pred'] = all_pred
mon_act  = df.groupby('MONTH')['DENGUE'].mean()
mon_pred = df.groupby('MONTH')['SVR_Pred'].mean()
months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(10,5))
xm = np.arange(1,13)
ax.plot(xm, mon_act,  'o-', color=P['test'],  lw=2.2, ms=7, label='Actual (Monthly Mean)')
ax.plot(xm, mon_pred, 's--',color=P['pred'],  lw=2.2, ms=7, label='SVR Predicted (Monthly Mean)')
ax.fill_between(xm, mon_act, mon_pred, alpha=0.12, color='grey', label='Difference')
ax.set_xticks(xm); ax.set_xticklabels(months)
ax.set_xlabel('Month'); ax.set_ylabel('Mean Monthly Dengue Cases')
ax.set_title('SVR (RBF) — Average Seasonal Dengue Pattern: Actual vs. Predicted\nBangladesh 2008–2025',
             fontweight='bold')
ax.legend(framealpha=0.85)
plt.tight_layout()
plt.savefig('/content/fig10_seasonal_pattern.png')
plt.close(); print("Fig 10 saved")

# Summary printout
print(f"""
╔══════════════════════════════════════════════════════════════════╗
║         SVR (RBF Kernel) — Final Model Summary                   ║
╠══════════════════════════════════════════════════════════════════╣
║  Kernel       : RBF                                              ║
║  Best C       : {bp['C']}                                        ║
║  Best gamma   : {bp['gamma']}                                    ║
║  Best epsilon : {bp['epsilon']}                                  ║
╠══════════════════════════════════════════════════════════════════╣
║  SPLIT        │   R²             │  RMSE               │   MAE              │  Pearson r           ║
║  Train (80%)  │ {m_tr['R2']:.4f} │ {m_tr['RMSE']:8.2f} │ {m_tr['MAE']:8.2f} │   {m_tr['r']:.4f}    ║
║  Test  (20%)  │ {m_te['R2']:.4f} │ {m_te['RMSE']:8.2f} │ {m_te['MAE']:8.2f} │   {m_te['r']:.4f}    ║
╠════════════════════════════════════════════════════════════════════════════════════════════════════╣
║  10-Fold CV R² (log scale): {cv_r2.mean():.4f} ± {cv_r2.std():.4f}                                 ║
╚════════════════════════════════════════════════════════════════════════════════════════════════════╝
""")
print(f"\nTest Set Sample (first 20 rows):\n{result_df.head(20).to_string(index=False)}")

Samples: 214  |  Train: 171 (80%)  |  Test: 43 (20%)
Best params: {'C': 5, 'epsilon': 0.01, 'gamma': 0.05}

── Metrics (original scale) ──
{'Set': 'Train 80%', 'R2': 0.9309, 'RMSE': np.float64(3025.81), 'MAE': 619.45, 'MAPE': np.float64(33.09), 'r': np.float64(0.9678), 'R2_log': 0.9426, 'RMSE_log': np.float64(0.728)}
{'Set': 'Test  20%', 'R2': 0.429, 'RMSE': np.float64(5776.94), 'MAE': 2168.12, 'MAPE': np.float64(100.42), 'r': np.float64(0.7317), 'R2_log': 0.8913, 'RMSE_log': np.float64(1.0306)}

10-Fold CV R² (log scale): 0.8244 ± 0.0809
Fig 1 saved
Fig 2 saved
Fig 3 saved
Fig 4 saved
Fig 5 saved
Fig 6 saved
Fig 7 saved
Fig 8 saved
Fig 9 saved
Fig 10 saved

╔══════════════════════════════════════════════════════════════════╗
║         SVR (RBF Kernel) — Final Model Summary                  ║
╠══════════════════════════════════════════════════════════════════╣
║  Kernel       : RBF                                             ║
║  Best C       : 5                                        